# Email Analysis

In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from spellchecker import SpellChecker

_SRC = Path("..").resolve() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

# talon 1.4.4 is incompatible with modern sklearn out of the box
# (sklearn.externals.joblib + pickled sklearn.svm.classes).
from email_cls_with_clustering.talon_compat import init_talon

signature, quotations = init_talon()
print("talon ready")

talon ready


In [3]:
import sys
from pathlib import Path

_SRC = Path("..").resolve() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from email_cls_with_clustering.tracking import setup_tracking

# Local SQLite store: ../mlflow.db. Pass enable_sklearn_autolog=True when training.
experiment = setup_tracking()
print(f"tracking uri ready → experiment={experiment.name} (id={experiment.experiment_id})")


/home/marco/development/email-cls-with-clustering/.venv/lib/python3.13/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.13) or chardet (7.6.0)/charset_normalizer (2.1.1) doesn't match a supported version!
  warnings.warn(


tracking uri ready → experiment=email-cls-with-clustering (id=1)


In [4]:
! kaggle datasets download wcukierski/enron-email-dataset

Dataset URL: https://www.kaggle.com/datasets/wcukierski/enron-email-dataset
License(s): copyright-authors
enron-email-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


## Preprocess Dataset

In [5]:
# load the dataset
# import zipfile

# with zipfile.ZipFile('enron-email-dataset.zip', 'r') as zip_ref:
#     zip_ref.extractall()

df = pd.read_csv('emails.csv')
df.head()

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   file     517401 non-null  str  
 1   message  517401 non-null  str  
dtypes: str(2)
memory usage: 1.3 GB


In [7]:
# from email_cls_with_clustering.preprocess import expand_emails

# df_expanded = expand_emails(df)

# preview_cols = ["From", "Subject", "body_raw", "body", "signature"]
# stripped = df_expanded["body_raw"].fillna("").ne(df_expanded["body"].fillna(""))
# preview = df_expanded.loc[stripped, preview_cols].head(3).copy()
# for col in ("body_raw", "body", "signature"):
#     preview[col] = preview[col].fillna("").str.slice(0, 400)
# preview

In [8]:
# df = df_expanded.copy()

In [9]:
# df = df.to_csv('emails_expanded.csv', index=False)

In [10]:
df = pd.read_csv('emails_expanded.csv')

/tmp/ipykernel_10560/4079247042.py:1: DtypeWarning: Columns (18: Time, 19: Attendees, 20: Re) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('emails_expanded.csv')


In [11]:
df.head()

,file,Message-ID,Date,From,To,Cc,Bcc,Subject,Mime-Version,Content-Type,...,Time,Attendees,Re,date_parsed,body_raw,body,signature,attachment_count,attachment_names,parse_error
0,allen-p/_sent_mail/1.,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700",phillip.allen@enron.com,tim.belden@enron.com,NaN,NaN,NaN,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2001-05-14 16:39:00-07:00,Here is our forecast,Here is our forecast,NaN,0,[],NaN
1,allen-p/_sent_mail/10.,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 04 May 2001 13:51:00 -0700",phillip.allen@enron.com,john.lavorato@enron.com,NaN,NaN,Re:,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2001-05-04 13:51:00-07:00,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...,NaN,0,[],NaN
2,allen-p/_sent_mail/100.,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700",phillip.allen@enron.com,leah.arsdall@enron.com,NaN,NaN,Re: test,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-10-18 03:00:00-07:00,test successful. way to go!!!,test successful. way to go!!!,NaN,0,[],NaN
3,allen-p/_sent_mail/1000.,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700",phillip.allen@enron.com,randall.gay@enron.com,NaN,NaN,NaN,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-10-23 06:13:00-07:00,"Randy,\n\n Can you send me a schedule of the s...","Randy,\n\n Can you send me a schedule of the s...",Phillip,0,[],NaN
4,allen-p/_sent_mail/1001.,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700",phillip.allen@enron.com,greg.piper@enron.com,NaN,NaN,Re: Hello,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-08-31 05:07:00-07:00,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.,NaN,0,[],NaN


## EDA

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 28 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   file                       517401 non-null  str    
 1   Message-ID                 517401 non-null  str    
 2   Date                       517401 non-null  str    
 3   From                       517401 non-null  str    
 4   To                         495554 non-null  str    
 5   Cc                         127881 non-null  str    
 6   Bcc                        127881 non-null  str    
 7   Subject                    498214 non-null  str    
 8   Mime-Version               517372 non-null  float64
 9   Content-Type               517372 non-null  str    
 10  Content-Transfer-Encoding  517372 non-null  str    
 11  X-From                     517372 non-null  str    
 12  X-To                       508248 non-null  str    
 13  X-cc                       128886 non-nu

In [13]:
df.columns

Index(['file', 'Message-ID', 'Date', 'From', 'To', 'Cc', 'Bcc', 'Subject',
       'Mime-Version', 'Content-Type', 'Content-Transfer-Encoding', 'X-From',
       'X-To', 'X-cc', 'X-bcc', 'X-Folder', 'X-Origin', 'X-FileName', 'Time',
       'Attendees', 'Re', 'date_parsed', 'body_raw', 'body', 'signature',
       'attachment_count', 'attachment_names', 'parse_error'],
      dtype='str')

In [14]:
df[df['To'].isna()].head()

,file,Message-ID,Date,From,To,Cc,Bcc,Subject,Mime-Version,Content-Type,...,Time,Attendees,Re,date_parsed,body_raw,body,signature,attachment_count,attachment_names,parse_error
188,allen-p/_sent_mail/264.,<15201149.1075855691021.JavaMail.evans@thyme>,"Mon, 01 May 2000 03:56:00 -0700",phillip.allen@enron.com,NaN,NaN,NaN,Re: DSL- Installs,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-05-01 03:56:00-07:00,No one will be home on 5/11/00 to meet DSL ins...,No one will be home on 5/11/00 to meet DSL ins...,"Call with questions. X37041.\n\nThank you,\n\n...",0,[],NaN
603,allen-p/all_documents/10.,<21975671.1075855665520.JavaMail.evans@thyme>,"Wed, 13 Dec 2000 08:35:00 -0800",messenger@ecm.bloomberg.com,NaN,NaN,NaN,Bloomberg Power Lines Report,1.0,"text/plain; charset=""ANSI_X3.4-1968""",...,NaN,NaN,NaN,2000-12-13 08:35:00-08:00,Here is today's copy of Bloomberg Power Lines....,Here is today's copy of Bloomberg Power Lines....,- daily.pdf,0,[],NaN
781,allen-p/all_documents/263.,<9828978.1075855671241.JavaMail.evans@thyme>,"Mon, 01 May 2000 03:56:00 -0700",phillip.allen@enron.com,NaN,NaN,NaN,Re: DSL- Installs,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-05-01 03:56:00-07:00,No one will be home on 5/11/00 to meet DSL ins...,No one will be home on 5/11/00 to meet DSL ins...,"Call with questions. X37041.\n\nThank you,\n\n...",0,[],NaN
873,allen-p/all_documents/348.,<8236042.1075855673105.JavaMail.evans@thyme>,"Fri, 07 Jan 2000 16:23:00 -0800",owner-strawbale@crest.org,NaN,NaN,NaN,NaN,1.0,"text/plain; charset=""us-ascii""",...,NaN,NaN,NaN,2000-01-07 16:23:00-08:00,<4DDE116DBCA1D3118B130080C840BAAD02CD53@ppims....,<4DDE116DBCA1D3118B130080C840BAAD02CD53@ppims....,NaN,0,[],NaN
885,allen-p/all_documents/359.,<26959382.1075855693279.JavaMail.evans@thyme>,"Mon, 14 May 2001 09:04:00 -0700",messenger@ecm.bloomberg.com,NaN,NaN,NaN,Bloomberg Power Lines Report,1.0,"text/plain; charset=""ANSI_X3.4-1968""",...,NaN,NaN,NaN,2001-05-14 09:04:00-07:00,Here is today's copy of Bloomberg Power Lines....,Here is today's copy of Bloomberg Power Lines....,- daily.pdf,0,[],NaN


In [15]:
df['Content-Type-no-charset'] = df['Content-Type'].str.split(';').str[0]
df['Content-Type-no-charset'].value_counts()

Content-Type-no-charset
text/plain    517372
Name: count, dtype: int64

In [16]:
df['attachment_count'].value_counts()

attachment_count
0    517401
Name: count, dtype: int64

In [17]:
# full_message using subject + body
df['full_message'] = df['Subject'] + ' ' + df['body']
df.duplicated(subset=['full_message']).sum()

np.int64(278270)

In [18]:
df = df.drop_duplicates(subset=['full_message'])
df.info()

<class 'pandas.DataFrame'>
Index: 239131 entries, 0 to 517400
Data columns (total 30 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   file                       239131 non-null  str    
 1   Message-ID                 239131 non-null  str    
 2   Date                       239131 non-null  str    
 3   From                       239131 non-null  str    
 4   To                         230770 non-null  str    
 5   Cc                         57801 non-null   str    
 6   Bcc                        57801 non-null   str    
 7   Subject                    239130 non-null  str    
 8   Mime-Version               239103 non-null  float64
 9   Content-Type               239103 non-null  str    
 10  Content-Transfer-Encoding  239103 non-null  str    
 11  X-From                     239103 non-null  str    
 12  X-To                       234174 non-null  str    
 13  X-cc                       58360 non-null   s

In [19]:
df['full_message'].iloc[1]

"Re: Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare a presentation.  I would suggest holding the business plan meetings here then take a trip without any formal business meetings.  I would even try and get some honest opinions on whether a trip is even desired or necessary.\n\nAs far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is not.  Too often the presenter speaks and the others are quiet just waiting for their turn.   The meetings might be better if held in a round table discussion format.  \n\nMy suggestion for where to go is Austin.  Play golf and rent a ski boat and jet ski's.  Flying somewhere takes too much time."

In [20]:
### Preprocessing for text-based analysis on the full_message
# spell = SpellChecker()

def preprocess_full_message(msg: str) -> str:
    if not isinstance(msg, str):
        return ""
    msg = msg.lower()

    # strip links and addresses while the punctuation they need is still present
    msg = re.sub(r'https?://\S+', '', msg)
    msg = re.sub(r'[\w.+-]+@[\w.-]+', '', msg)

    # require a real digit grouping; whitespace alone must not match
    msg = re.sub(
        r'(?:\+?\d{1,3}[\s.-])?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}|\b\d{10,}\b',
        '',
        msg,
    )

    # remove noisy symbols (more than 1 dashes or underscores)
    msg = re.sub(r'[-_]{2,}', '', msg)

    msg = re.sub(r'[^a-z0-9\s]', '', msg)
    msg = re.sub(r'\s+', ' ', msg).strip()

    return msg

df['full_message'] = df['full_message'].apply(preprocess_full_message)

In [21]:
df['word_count'] = df['full_message'].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)
df['word_count'].describe()

count    239131.000000
mean        163.836144
std         507.093634
min           0.000000
25%          20.000000
50%          50.000000
75%         135.000000
max       25173.000000
Name: word_count, dtype: float64

## BERTopic Application and Analysis

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

# Default backend for embed_texts() / cache naming. Override per-call via backend=.
EMBED_BACKEND = "sentence_transformer"  # "sentence_transformer" | "openai"

ST_MODEL = "ibm-granite/granite-embedding-97m-multilingual-r2"
ST_MAX_SEQ_LENGTH = 2048  # safe on Radeon 8060S (~14GB); model supports 32k. Try 4096 if you want.
ST_BATCH_SIZE = 32  # lower than 64 so longer sequences stay comfortable on ROCm

OPENAI_MODEL = "text-embedding-3-small"  # or "text-embedding-3-small"
OPENAI_BATCH_SIZE = 64
OPENAI_MAX_TOKENS = 8191

_st_model = None
_openai_client = None
_tiktoken_enc = None


def _ensure_sentence_transformer():
    global _st_model
    if _st_model is None:
        from sentence_transformers import SentenceTransformer

        _st_model = SentenceTransformer(
            ST_MODEL,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
        _st_model.max_seq_length = ST_MAX_SEQ_LENGTH
        device = next(_st_model.parameters()).device
        print(
            f"loaded sentence_transformer model={ST_MODEL} "
            f"dim={_st_model.get_embedding_dimension()} "
            f"device={device} max_seq={_st_model.max_seq_length}"
        )
    return _st_model


def _ensure_openai():
    global _openai_client, _tiktoken_enc
    if _openai_client is None:
        import tiktoken
        from dotenv import load_dotenv
        from openai import OpenAI

        load_dotenv(Path(".env"))
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY missing — set it in notebooks/.env")

        _openai_client = OpenAI()
        _tiktoken_enc = tiktoken.get_encoding("cl100k_base")
        print(f"loaded openai client model={OPENAI_MODEL}")
    return _openai_client, _tiktoken_enc


def active_embed_model(backend: str | None = None) -> str:
    backend = backend or EMBED_BACKEND
    if backend == "sentence_transformer":
        return ST_MODEL
    if backend == "openai":
        return OPENAI_MODEL
    raise ValueError(f"Unknown backend={backend!r}; use 'sentence_transformer' or 'openai'")


def _clean_texts(texts: list[str]) -> list[str]:
    return [(t or "").strip() or " " for t in texts]


def _truncate_openai(text: str, enc, max_tokens: int = OPENAI_MAX_TOKENS) -> str:
    tokens = enc.encode(text, disallowed_special=())
    if len(tokens) <= max_tokens:
        return text
    return enc.decode(tokens[:max_tokens])


def embed_texts(
    texts: list[str],
    backend: str | None = None,
    batch_size: int | None = None,
) -> np.ndarray:
    """Embed texts with sentence-transformers or OpenAI.

    backend: "sentence_transformer" | "openai" | None → defaults to EMBED_BACKEND
    """
    backend = backend or EMBED_BACKEND
    cleaned = _clean_texts(texts)

    if backend == "sentence_transformer":
        st_model = _ensure_sentence_transformer()
        bs = batch_size or ST_BATCH_SIZE
        return st_model.encode(
            cleaned,
            batch_size=bs,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)

    if backend == "openai":
        client, enc = _ensure_openai()
        bs = batch_size or OPENAI_BATCH_SIZE
        truncated = [_truncate_openai(t, enc) for t in cleaned]
        vectors: list[list[float]] = []
        for start in tqdm(range(0, len(truncated), bs), desc=f"embed ({OPENAI_MODEL})"):
            batch = truncated[start : start + bs]
            response = client.embeddings.create(model=OPENAI_MODEL, input=batch)
            ordered = sorted(response.data, key=lambda row: row.index)
            vectors.extend(row.embedding for row in ordered)
        return np.asarray(vectors, dtype=np.float32)

    raise ValueError(f"Unknown backend={backend!r}; use 'sentence_transformer' or 'openai'")


EMBED_MODEL = active_embed_model()
print(f"default backend={EMBED_BACKEND} model={EMBED_MODEL}")

default backend=sentence_transformer model=ibm-granite/granite-embedding-97m-multilingual-r2


In [23]:
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.en.stop_words import STOP_WORDS as EN_STOP
from spacy.lang.es.stop_words import STOP_WORDS as ES_STOP
from umap import UMAP

texts = df["full_message"].fillna("").astype(str).tolist()

# Cache keyed by backend + model so switching doesn't reuse mismatched dims
embed_cache = Path(
    f"embeddings_{EMBED_BACKEND}_{active_embed_model().replace('/', '_')}.npy"
)
if embed_cache.exists():
    embeddings = np.load(embed_cache)
    if len(embeddings) != len(texts):
        raise ValueError(
            f"Cache length {len(embeddings)} != {len(texts)} texts — delete {embed_cache} and re-run"
        )
    print(f"loaded embeddings from {embed_cache} → shape={embeddings.shape}")
else:
    if EMBED_BACKEND == "sentence_transformer" and torch.cuda.is_available():
        torch.cuda.empty_cache()
    embeddings = embed_texts(texts)  # or embed_texts(texts, backend="openai")
    np.save(embed_cache, embeddings)
    print(f"saved embeddings → {embed_cache} shape={embeddings.shape}")

# Precomputed embeddings → no default MiniLM; cosine UMAP matches L2-normalized vectors.
# min_cluster_size=50 is more appropriate than the default (~10) for ~200k+ emails.
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    prediction_data=True,  # required for soft-cluster probabilities
)

stop_words = list(EN_STOP | ES_STOP)
vectorizer_model = CountVectorizer(stop_words=stop_words)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True,
)
topics, probs = topic_model.fit_transform(texts, embeddings)

df["topic"] = topics
df["prob_vector"] = list(probs)

df["topic"].value_counts().head(15)

2026-09-23 16:48:07,404 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


loaded embeddings from embeddings_sentence_transformer_ibm-granite_granite-embedding-97m-multilingual-r2.npy → shape=(239131, 384)


2026-09-23 16:51:46,560 - BERTopic - Dimensionality - Completed ✓
2026-09-23 16:51:46,562 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-23 18:06:10,723 - BERTopic - Cluster - Completed ✓
2026-09-23 18:06:10,826 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 18:06:31,137 - BERTopic - Representation - Completed ✓


topic
-1     132569
 0       3224
 1       1779
 2       1694
 3       1638
 4       1406
 5       1399
 6       1309
 7       1144
 8       1101
 9       1095
 10      1069
 11      1055
 12      1014
 13       996
Name: count, dtype: int64

In [21]:
df.to_csv('emails_clustered.csv', index=False)

In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 239131 entries, 0 to 517400
Data columns (total 30 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   file                       239131 non-null  str    
 1   Message-ID                 239131 non-null  str    
 2   Date                       239131 non-null  str    
 3   From                       239131 non-null  str    
 4   To                         230770 non-null  str    
 5   Cc                         57801 non-null   str    
 6   Bcc                        57801 non-null   str    
 7   Subject                    239130 non-null  str    
 8   Mime-Version               239103 non-null  float64
 9   Content-Type               239103 non-null  str    
 10  Content-Transfer-Encoding  239103 non-null  str    
 11  X-From                     239103 non-null  str    
 12  X-To                       234174 non-null  str    
 13  X-cc                       58360 non-null   s

In [24]:
plt.bar(df['topic'].value_counts().index, df['topic'].value_counts().values)
plt.show()

KeyError: 'topic'

In [ ]:
freq = topic_model.get_topic_info()
freq.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,132569,-1_know_subject_email_thanks,"[know, subject, email, thanks, cc, pm, message...",[ferc issues for enaees per your request here ...
1,0,3224,0_meeting_meetings_agenda_attend,"[meeting, meetings, agenda, attend, staff, boa...",[staff meeting there will be a friday staff me...
2,1,1779,1_draft_letter_comments_tva,"[draft, letter, comments, tva, intended, revis...","[re final draft done, dow letter attached is t..."
3,2,1694,2_davis_state_california_electricity,"[davis, state, california, electricity, utilit...",[energy issues please see the following articl...
4,3,1638,3_checkout_kate_deal_apb,"[checkout, kate, deal, apb, prebon, broker, de...",[re 38 checkout thanks kate symes ect 03082001...
5,4,1406,4_testimony_subpoena_deposition_plaintiffs,"[testimony, subpoena, deposition, plaintiffs, ...",[nsm update forwarded by richard b sandershoue...
6,5,1399,5_gas_logistics_natural_deal,"[gas, logistics, natural, deal, ces, volume, d...",[enerfaxdaily natural gas futures settle sligh...
7,6,1309,6_yes_dont_yep_yeah,"[yes, dont, yep, yeah, im, ok, whats, sorry, h...","[re yes you are, re yes everything, re yes and..."
8,7,1144,7_donate_declared_consumers_employees,"[donate, declared, consumers, employees, retir...",[demand ken lay donate proceeds from enron sto...
9,8,1101,8_isda_master_sara_ibj,"[isda, master, sara, ibj, agreement, annex, dr...",[master isda agreement sara i have spoken to k...


In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.visualize_barchart()